In [1]:
def calc_normalizer_regs(
    min_count: float,
    max_count: float,
    desired_peak: int = 0x70000000,
):
    """
    AD7768 average/source 값을 PID signed-32 입력 범위로 정규화하기 위한
    center / scale 레지스터 값을 계산한다.

    FPGA 내부 normalizer 식:

        pid_input = saturate32(
            ((source - center_reg) * scale_reg) >> 16
        )

    여기서:
        source      : signed 32-bit 컨테이너
                      실제 유효 데이터는 AD7768 signed 24-bit count
        center_reg  : signed 32-bit integer
        scale_reg   : signed Q16.16
        pid_input   : signed 32-bit

    Parameters
    ----------
    min_count : float
        실제 사용할 입력 범위의 최소값.
        ADC count 기준으로 넣으면 된다.

        예:
            -500000.0

    max_count : float
        실제 사용할 입력 범위의 최대값.

        예:
            +700000.0

    desired_peak : int
        정규화 후 사용할 signed32 최대 크기.

        기본값:
            0x70000000

        0x7FFFFFFF까지 꽉 채우지 않고 약간의 headroom을 두기 위해
        0x70000000을 기본값으로 사용한다.

    Returns
    -------
    center_reg : int
        FPGA center 레지스터에 쓸 signed32 정수값

    scale_reg : int
        FPGA scale 레지스터에 쓸 signed Q16.16 정수값

    scale_real : float
        사람이 보기 위한 실제 배율
    """

    if max_count <= min_count:
        raise ValueError("max_count must be greater than min_count")

    # ------------------------------------------------------------
    # 1. 입력 범위의 중심값 계산
    #
    # 예:
    #   min = -500000
    #   max = +700000
    #
    #   center = (+700000 - 500000) / 2
    #          = 100000
    #
    # 즉 source에서 100000을 빼면 입력 범위가
    # -600000 ~ +600000으로 0 중심이 된다.
    # ------------------------------------------------------------
    center_real = (max_count + min_count) / 2.0

    # FPGA center register는 정수이므로 마지막에 반올림
    center_reg = round(center_real)

    # ------------------------------------------------------------
    # 2. 중심으로부터 최대 입력 크기 계산
    #
    # 위 예에서는:
    #
    #   half_range = (700000 - (-500000)) / 2
    #              = 600000
    #
    # center를 뺀 뒤 입력 범위는 대략:
    #
    #   -600000 ~ +600000
    # ------------------------------------------------------------
    half_range = (max_count - min_count) / 2.0

    # ------------------------------------------------------------
    # 3. 실제 필요한 gain 계산
    #
    # half_range가 desired_peak에 대응하도록 만든다.
    #
    # 즉:
    #
    #   +half_range -> +desired_peak
    #   0           -> 0
    #   -half_range -> -desired_peak
    #
    # 따라서:
    #
    #   scale_real = desired_peak / half_range
    # ------------------------------------------------------------
    scale_real = desired_peak / half_range

    # ------------------------------------------------------------
    # 4. scale을 FPGA Q16.16 형식으로 변환
    #
    # Q16.16에서는:
    #
    #   1.0   -> 0x00010000
    #   0.5   -> 0x00008000
    #   2.0   -> 0x00020000
    #   256.0 -> 0x01000000
    #
    # 따라서 실제 gain에 2^16을 곱해서 정수로 저장한다.
    # ------------------------------------------------------------
    scale_reg = round(scale_real * (1 << 16))

    # signed 32-bit Q16.16에 들어가는지 확인
    if not (-0x80000000 <= scale_reg <= 0x7FFFFFFF):
        raise ValueError(
            f"scale_reg does not fit signed 32-bit Q16.16: {scale_reg}"
        )

    if not (-0x80000000 <= center_reg <= 0x7FFFFFFF):
        raise ValueError(
            f"center_reg does not fit signed 32-bit: {center_reg}"
        )

    return center_reg, scale_reg, scale_real


# ============================================================
# 사용 예
# ============================================================

min_count = -500000.0
max_count = +700000.0

center_reg, scale_reg, scale_real = calc_normalizer_regs(
    min_count,
    max_count,
)

print(f"center_reg = {center_reg}")
print(f"center_reg hex = 0x{center_reg & 0xFFFFFFFF:08X}")

print(f"scale_real = {scale_real:.6f}")
print(f"scale_reg = {scale_reg}")
print(f"scale_reg hex = 0x{scale_reg & 0xFFFFFFFF:08X}")

center_reg = 100000
center_reg hex = 0x000186A0
scale_real = 3131.746987
scale_reg = 205242171
scale_reg hex = 0x0C3BBF3B


In [2]:
def normalize_like_fpga(source: int, center_reg: int, scale_reg: int) -> int:
    """
    FPGA input_normalizer의 계산을 Python에서 흉내낸다.

        y = ((source - center) * scale) >> 16

    마지막에 signed32 saturation 수행.
    """

    value = ((source - center_reg) * scale_reg) >> 16

    # signed 32-bit saturation
    if value > 0x7FFFFFFF:
        value = 0x7FFFFFFF
    elif value < -0x80000000:
        value = -0x80000000

    return value